# Phase-D: hybrid K=10 epoch-curve eval on Phase-B ckpts**Goal**: show the headline claim that **MBD can lift undertrainedDP priors to match the fully-trained one**.  We re-evaluate thev3b training-curve checkpoints (ep0049, ep0099, ep0149, ep0199, ...)under the **hybrid** pipeline (DP prior + MBD K=10) and compare tothe standalone DP numbers already in Drive (Phase-B notebook cell 20).Story expected:| ckpt | standalone (DP only) | hybrid (DP + MBD K=10) | lift ||---|---:|---:|---:|| ep0049 | 46 % | **TODO (hypothesis: ~70-80 %)** | **+25-35 pp** || ep0099 | 66 % | TODO (~75-80 %) | +10-15 pp || ep0149 | 64 % | TODO | TODO || ep0199 | **80 %** | TODO (plateau or +0-5 pp) | TODO |If hybrid at ep0049 lands near 75-80 %, the headline reads:**"MBD makes undertrained DP indistinguishable from fully-trained DP."**### Pipeline correctnessThis notebook clones the `feat/dp-phase-c` branch of HRTG-dp,which includes the **Phase-D fix** (commit `a58af13`) for thechunk-boundary `env.reset()` bug that previously caused 0 % at K=0in the unified driver.  Cell 15 (sanity grep) verifies the fix ispresent before running any eval.### Wall budget (Colab CPU, 2 vCPU free tier)| K=10 hybrid eval per ckpt | ~90-140 min || 4 ckpts (ep0049/0099/0149/0199), sequential | ~6-9 hr || 4 ckpts in parallel (open 4 Colab tabs, set `CKPT_FILTER` per tab) | ~90-140 min |Idempotency: each ckpt's hybrid result is cached to Drive.Re-running the notebook (in the same or a different Colab session)skips ckpts whose result JSON is already complete.

## 1. Confirm CPU info (we don't need GPU for hybrid eval; env stepping dominates)

In [ ]:

!cat /proc/cpuinfo | grep 'model name' | head -1
!nproc
!free -h


## 2. Verify PyTorch + pin numpy 2.0.2 (robosuite compatibility)

In [ ]:

import subprocess, sys

# Pin numpy to 2.0.2 BEFORE anything else imports it.
# robosuite 1.5.2 + mujoco 3.8.1 require numpy 2.0.x.
# Colab ships numpy 1.26.x by default.
import importlib
try:
    import numpy as np
    print(f"  numpy (pre-pin): {np.__version__}")
    needs_pin = not np.__version__.startswith("2.0")
except ImportError:
    needs_pin = True

if needs_pin:
    print("  installing numpy==2.0.2 (~30 s) ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--force-reinstall", "numpy==2.0.2",
    ])
    print("  RESTART RUNTIME and re-run this cell.")
    import os
    os.kill(os.getpid(), 9)
else:
    import torch
    print(f"  torch    : {torch.__version__}")
    print(f"  cuda?    : {torch.cuda.is_available()}")


## 3. Mount Drive + define pathsWe re-use the Phase-B notebook's Drive folder(`MyDrive/hybrid_traj_dp_phase_b/`) which contains both thetraining ckpts and the standalone eval JSONs.

In [ ]:

from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/hybrid_traj_dp_phase_b"
assert os.path.isdir(SAVE_DIR), (
    f"Phase-B Drive folder missing: {SAVE_DIR}\n"
    f"This notebook requires the Phase-B training ckpts (the same\n"
    f"`colab_train_dp_phase_b.ipynb` notebook produced).  Either:\n"
    f"  (a) Run that notebook first (cell 14 trains and saves\n"
    f"      ckpts to Drive), or\n"
    f"  (b) Change SAVE_DIR above to wherever the Phase-B ckpts live."
)

# Sub-folders this notebook reads / writes.
CKPT_DIR = SAVE_DIR                                              # ckpts at top level
STANDALONE_EVAL_DIR = f"{SAVE_DIR}/standalone_eval_phase_b"      # READ-ONLY (Phase-B output)
HYBRID_EVAL_DIR = f"{SAVE_DIR}/hybrid_eval_phase_b"              # WRITE (this notebook)
os.makedirs(HYBRID_EVAL_DIR, exist_ok=True)

REPO_DIR = "/content/HRTG-dp"
DATA_DIR = "/content/can_ph"
DATA_PATH = f"{DATA_DIR}/low_dim_v141.hdf5"

print(f"SAVE_DIR             = {SAVE_DIR}")
print(f"CKPT_DIR             = {CKPT_DIR}")
print(f"STANDALONE_EVAL_DIR  = {STANDALONE_EVAL_DIR}  (read-only)")
print(f"HYBRID_EVAL_DIR      = {HYBRID_EVAL_DIR}  (write)")
print(f"REPO_DIR             = {REPO_DIR}")
print(f"DATA_PATH            = {DATA_PATH}")
print()
print(f"ckpts available:")
import glob
for p in sorted(glob.glob(f"{CKPT_DIR}/v_theta_dp_tobs2_h16_v3b*.pt")):
    print(f"  {os.path.basename(p):50s}  {os.path.getsize(p)/1e6:>7.1f} MB")
print()
print(f"standalone results (already computed by Phase-B notebook):")
for p in sorted(glob.glob(f"{STANDALONE_EVAL_DIR}/*/standalone_*.json")):
    print(f"  {p.replace(STANDALONE_EVAL_DIR + '/', '')}")


## 4. Clone HRTG-dp at `feat/dp-phase-c` (the post-Phase-D branch)We need the `feat/dp-phase-c` branch because it contains thePhase-D fix to `unified_loop.py` (commit `a58af13`).  Withoutthis, the K=0 path returns 0 % and the K>0 path is biased.

In [ ]:

import os, subprocess

# Use https with anonymous read; public repo (or yours if private,
# in which case set GH_TOKEN below).
REPO_URL = "https://github.com/watamelon2/Hybrid-Robot-Trajectory-Generation.git"
BRANCH = "feat/dp-phase-c"

if os.path.isdir(REPO_DIR):
    print(f"[update] {REPO_DIR} exists; pulling latest from {BRANCH}")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    print(f"[clone] {REPO_URL}@{BRANCH} -> {REPO_DIR}")
    subprocess.check_call([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, REPO_DIR,
    ])

# Show the head commit so we know exactly which code is running.
subprocess.check_call(["git", "-C", REPO_DIR, "log", "--oneline", "-3"])
print()

# CRITICAL: the Phase-D fix must be present.  Verify by grepping
# the gate condition in unified_loop.py.
print("[verify] Phase-D fix (chunk-boundary reset gate)...")
subprocess.check_call([
    "bash", "-c",
    f"grep -n 'params.K' {REPO_DIR}/experiments/15_unified_hybrid/unified_loop.py "
    f"| grep -i '> 0' || (echo 'PHASE-D FIX MISSING!!' && exit 1)"
])
print("[ok] Phase-D fix present.")


## 5. Procure dataset (re-use Phase-B's Drive cache)

In [ ]:

import os, shutil, subprocess, sys

# Same priority list as the Phase-B notebook's cell 10.
DRIVE_CANDIDATES = [
    f"{SAVE_DIR}/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase3p5/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase2b/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase2/low_dim_v141.hdf5",
]

try:
    import robomimic
    print(f"robomimic: {robomimic.__version__}")
except ImportError:
    print("installing robomimic (~30 s)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "robomimic==0.3.0"])
    import robomimic

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.isfile(DATA_PATH):
    for cand in DRIVE_CANDIDATES:
        if os.path.isfile(cand) and os.path.getsize(cand) > 1_000_000:
            print(f"[1] using Drive copy at {cand}")
            shutil.copy(cand, DATA_PATH)
            break

assert os.path.isfile(DATA_PATH), (
    f"No dataset at {DATA_PATH} and no Drive copy found.\n"
    f"Run the Phase-B notebook first (it caches the dataset to Drive)."
)
print(f"dataset OK: {DATA_PATH}  size={os.path.getsize(DATA_PATH) / 1e6:.1f} MB")

# Symlink into robomimic's expected dataset path.
from pathlib import Path
pkg_dir = Path(robomimic.__file__).resolve().parent.parent
target_dir = pkg_dir / "datasets" / "can" / "ph"
target_dir.mkdir(parents=True, exist_ok=True)
sym_target = target_dir / "low_dim_v141.hdf5"
if sym_target.is_symlink() or sym_target.exists():
    sym_target.unlink()
sym_target.symlink_to(DATA_PATH)
print(f"symlink: {sym_target} -> {DATA_PATH}")


## 6. Install robosuite + mujoco (the env stack)

In [ ]:

import subprocess, sys

try:
    import numpy as np, robosuite
    if np.__version__.startswith("2.0"):
        print(f"  numpy    : {np.__version__}")
        print(f"  robosuite: {robosuite.__version__}")
        SKIP = True
    else:
        SKIP = False
except (ImportError, ValueError):
    SKIP = False

if not SKIP:
    print("[1/3] installing robosuite + mujoco (~2 min) ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "robosuite==1.5.2", "mujoco==3.8.1",
    ])
    print("[2/3] re-pinning numpy==2.0.2 (robosuite install downgrades it) ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--force-reinstall", "numpy==2.0.2",
    ])
    print("[3/3] importing ...")
    import numpy as np, robosuite
    print(f"  numpy    : {np.__version__}")
    print(f"  robosuite: {robosuite.__version__}")
    print("  OK")


## 7. Quick sanity: import the unified driver

In [ ]:

import sys, importlib

# Make the repo's experiments/ importable.
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/experiments")
sys.path.insert(0, f"{REPO_DIR}/experiments/15_unified_hybrid")
sys.path.insert(0, f"{REPO_DIR}/experiments/04_mbd_baseline")
sys.path.insert(0, f"{REPO_DIR}/experiments/03_bc_baseline")
sys.path.insert(0, f"{REPO_DIR}/experiments/08_flow_matching_prior")

# Light probe: can we import the unified plan-and-execute?
import importlib.util
spec = importlib.util.spec_from_file_location(
    "unified_loop",
    f"{REPO_DIR}/experiments/15_unified_hybrid/unified_loop.py",
)
unified_loop = importlib.util.module_from_spec(spec)
spec.loader.exec_module(unified_loop)
print(f"unified_plan_and_execute: {unified_loop.unified_plan_and_execute}")
print("OK")


## 8. Hybrid K=10 eval across the v3b epoch ckpts (THE MAIN CELL)This runs the same hybrid pipeline as the local Phase-C K-sweep(`run_phase_c_ksweep.sh`), one ckpt at a time, with idempotentDrive caching.**To parallelize across multiple Colab tabs**: set `CKPT_FILTER`to a substring of the ckpt name you want this tab to handle.For example, open 4 Colab tabs and set:  - tab 1: `CKPT_FILTER = "ep0049"`  - tab 2: `CKPT_FILTER = "ep0099"`  - tab 3: `CKPT_FILTER = "ep0149"`  - tab 4: `CKPT_FILTER = "ep0199"`Each tab will skip the others' work (Drive caching is shared) sothey don't conflict.

In [ ]:

import subprocess, sys, os, json, time, glob, shutil
from pathlib import Path

# === Hybrid pipeline knobs (must match local run_phase_c_ksweep.sh) ===
K = 10                                 # MBD denoise depth -- the active knob
LAMBDA_PRIOR = 0.5                     # prior score weight (Phase-C fixed point)
N_DEMOS, N_ROLLOUTS = 10, 1            # 10 train-init replays
N_TEST_ROLLOUTS = 50                   # 50 fresh-seed test rollouts
TEST_START_SEED = 100_000              # DP paper default
H, MPC_REPLANS, MPC_CHUNK_STEPS = 16, 49, 8
MAX_ENV_STEPS = 400
SIGMA_MAX, SIGMA_MIN, PRIOR_N_STEPS = 0.25, 0.05, 100
N_WORKERS = 2                          # Phase-D fix: subprocess workers own forked envs

# === Filter: which ckpts to evaluate in THIS Colab tab ===
# Empty string = all v3b ckpts (sequential).  Set to e.g. "ep0049"
# to make this tab handle only one ckpt (parallelize across tabs).
CKPT_FILTER = ""

# === Candidate ckpts ===
candidates = sorted(glob.glob(f"{CKPT_DIR}/v_theta_dp_tobs2_h16_v3b*.pt"))
if CKPT_FILTER:
    candidates = [c for c in candidates if CKPT_FILTER in os.path.basename(c)]
assert candidates, f"no ckpts match filter {CKPT_FILTER!r} in {CKPT_DIR}"
print(f"this tab will evaluate {len(candidates)} ckpt(s) "
      f"(filter={CKPT_FILTER!r}):")
for c in candidates:
    print(f"  {os.path.basename(c):50s}  {os.path.getsize(c)/1e6:>7.1f} MB")

# === Eval loop with Drive caching ===
results = []
for ckpt_path in candidates:
    tag = Path(ckpt_path).stem               # e.g. v_theta_dp_tobs2_h16_v3b_ep0099
    drive_ckpt_dir = f"{HYBRID_EVAL_DIR}/{tag}"
    os.makedirs(drive_ckpt_dir, exist_ok=True)
    # Match the local launcher's filename: unified_flow_lam-0.5_K-10_R-49.json
    out_name = f"unified_flow_lam-{LAMBDA_PRIOR}_K-{K}_R-{MPC_REPLANS}.json"
    drive_out = f"{drive_ckpt_dir}/{out_name}"

    # Idempotency: skip if Drive has a complete result.
    if os.path.isfile(drive_out):
        try:
            d = json.load(open(drive_out))
            if int(d.get("n_total", 0)) == (N_DEMOS * N_ROLLOUTS + N_TEST_ROLLOUTS):
                print(f"[skip] {tag}  (Drive cached: {drive_out})")
                results.append({"tag": tag, "result": d})
                continue
        except Exception as e:
            print(f"[warn] could not parse cached {drive_out}: {e}; re-eval")

    print(f"\n{'='*72}\n[eval] {tag}  K={K}  lam={LAMBDA_PRIOR}\n{'='*72}")
    t0 = time.time()
    proc = subprocess.Popen([
        sys.executable, "-u",
        "-m", "experiments.15_unified_hybrid.run_can",
        "--task", "can",
        "--prior", "flow",
        "--prior-checkpoint", ckpt_path,
        "--H", str(H),
        "--mpc-replans", str(MPC_REPLANS),
        "--mpc-chunk-steps", str(MPC_CHUNK_STEPS),
        "--n-demos", str(N_DEMOS),
        "--n-rollouts", str(N_ROLLOUTS),
        "--n-test-rollouts", str(N_TEST_ROLLOUTS),
        "--test-start-seed", str(TEST_START_SEED),
        "--max-env-steps", str(MAX_ENV_STEPS),
        "--prior-n-steps", str(PRIOR_N_STEPS),
        "--sigma-max", str(SIGMA_MAX),
        "--sigma-min", str(SIGMA_MIN),
        "--lambda-prior", str(LAMBDA_PRIOR),
        "--device", "cpu",
        "--seed", "0",
        "--K", str(K),
        "--n-workers", str(N_WORKERS),
        "--output-dir", drive_ckpt_dir,
    ], cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream a filtered view of the output (the [exp15] rollout
    # lines + key milestones) so the Colab cell shows progress.
    for line in proc.stdout:
        if any(tok in line for tok in ("[exp15]", "[ksweep]", "error", "Error", "Traceback")):
            print(line.rstrip())
    rc = proc.wait()
    dt = time.time() - t0
    print(f"\n[done] rc={rc}  wall={dt/60:.1f} min")
    if rc != 0:
        print(f"[ERR ] eval failed for {tag}; check the full log in the cell output above.")
        continue

    # Load + cache the result JSON.
    local_out = f"{drive_ckpt_dir}/{out_name}"
    if os.path.isfile(local_out):
        d = json.load(open(local_out))
        print(f"  train: {d.get('train_n_succ', '?')}/{d.get('train_n', '?')}")
        print(f"  test : {d.get('test_n_succ', '?')}/{d.get('test_n', '?')}")
        print(f"  total: {d.get('n_success', '?')}/{d.get('n_total', '?')} = "
              f"{float(d.get('success_rate', 0))*100:.1f} %")
        results.append({"tag": tag, "result": d})
    else:
        print(f"[WARN] expected output {local_out} not found")

print(f"\n{'='*72}\nthis tab finished: {len(results)} ckpt result(s) recorded\n{'='*72}")


## 9. Aggregate standalone + hybrid, print side-by-side tableReads BOTH the standalone results (from Phase-B notebook) and thehybrid results (this notebook).  Prints a single comparison tablecovering whichever ckpts have results on Drive.

In [ ]:

import os, json, glob, re
from pathlib import Path

def _epoch_of(tag: str) -> int:
    m = re.search(r"ep(\d+)", tag)
    return int(m.group(1)) if m else 0

def _load_standalone(tag: str):
    path = f"{STANDALONE_EVAL_DIR}/{tag}/standalone_{tag}.json"
    if not os.path.isfile(path):
        return None
    d = json.load(open(path))
    return {
        "train": float(d.get("train_succ_rate", 0.0)),
        "test":  float(d.get("test_succ_rate", 0.0)),
        "total": float(d.get("succ_rate", 0.0)),
        "n_train": int(d.get("train_n", 0)),
        "n_test":  int(d.get("test_n", 0)),
        "n_total": int(d.get("n", 0)),
    }

def _load_hybrid(tag: str):
    # Match local naming: unified_flow_lam-0.5_K-10_R-49.json
    path_pattern = f"{HYBRID_EVAL_DIR}/{tag}/unified_flow_lam-*_K-*_R-*.json"
    paths = sorted(glob.glob(path_pattern))
    if not paths:
        return None
    d = json.load(open(paths[0]))
    return {
        "train": float(d.get("train_succ_rate", 0.0)),
        "test":  float(d.get("test_succ_rate", 0.0)),
        "total": float(d.get("success_rate", 0.0)),
        "n_train": int(d.get("train_n", 0)),
        "n_test":  int(d.get("test_n", 0)),
        "n_total": int(d.get("n_total", 0)),
        "wall_s": float(d.get("wall_s", 0.0)),
    }

# Union of all v3b ckpts seen in either dir.
tags = set()
for p in glob.glob(f"{STANDALONE_EVAL_DIR}/*/standalone_*.json"):
    tags.add(Path(p).stem.replace("standalone_", ""))
for p in glob.glob(f"{HYBRID_EVAL_DIR}/v_theta_dp_tobs2_h16_v3b*"):
    tags.add(os.path.basename(p))

rows = []
for tag in sorted(tags, key=_epoch_of):
    s = _load_standalone(tag)
    h = _load_hybrid(tag)
    rows.append({"tag": tag, "epoch": _epoch_of(tag), "standalone": s, "hybrid": h})

print("=" * 92)
print(f"{'ckpt':<42s}  {'epoch':>5}  {'stand test':>10}  {'hybrid test':>11}  {'lift':>7}")
print("=" * 92)
for r in rows:
    epoch = r["epoch"]
    s_test = r["standalone"]["test"] if r["standalone"] else None
    h_test = r["hybrid"]["test"] if r["hybrid"] else None
    s_str = f"{s_test*100:5.1f} %" if s_test is not None else "  ---  "
    h_str = f"{h_test*100:5.1f} %" if h_test is not None else "  ---  "
    if s_test is not None and h_test is not None:
        lift = (h_test - s_test) * 100
        l_str = f"{lift:+5.1f} pp"
    else:
        l_str = "   ---  "
    print(f"  {r['tag']:<40s}  {epoch:>5d}     {s_str}        {h_str}    {l_str}")
print("=" * 92)


## 10. Plot the learning curves (standalone vs hybrid)

In [ ]:

import matplotlib.pyplot as plt

xs = [r["epoch"] for r in rows if r["standalone"] is not None]
ys_stand = [r["standalone"]["test"] * 100 for r in rows if r["standalone"] is not None]
xs_h = [r["epoch"] for r in rows if r["hybrid"] is not None]
ys_hybr = [r["hybrid"]["test"] * 100 for r in rows if r["hybrid"] is not None]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, ys_stand, "o-", label="standalone DP (DDPM K=100)", linewidth=2, markersize=8)
if xs_h:
    ax.plot(xs_h, ys_hybr, "s-", color="C1",
            label=f"hybrid DP + MBD (K=10, λ=0.5)", linewidth=2, markersize=8)
ax.set_xlabel("Phase-B training epoch")
ax.set_ylabel("test success rate (%, 50 fresh seeds)")
ax.set_title("Can/ph: standalone DP vs DP + MBD across v3b training")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

# Also save to Drive for the BRIEF.
out_png = f"{HYBRID_EVAL_DIR}/epoch_curve_standalone_vs_hybrid.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"saved plot to {out_png}")


## 11. Next steps on local MacIf the hybrid curve shows large lifts on undertrained ckpts(especially ep0049: standalone ~46 % → hybrid ~75-80 %), that'sthe headline result for Phase-D / Phase-C.After this notebook finishes:1. Pull the result JSONs from Drive to local:       python experiments/08_flow_matching_prior/sync_ckpts_from_drive.py \\           --folder-url 'https://drive.google.com/drive/folders/<HYBRID_EVAL_DIR_ID>?usp=sharing' \\           --pattern unified_flow_2. Update `experiments/17_dp_hybrid/results/phase_d_2026_05_30/CONCLUSION.md`   with the headline numbers + plot.3. Tag the branch and merge `feat/dp-phase-c` → `main`.